## Import

**Amaç:** `src` klasörünü Python path'ine eklemek ve `silver_transforms` ile `silver_merge` modüllerini yüklemek.

**Not:** `.py` dosyası değişince bu hücre tekrar çalıştırılır (`importlib.reload`).

In [0]:
import os
import sys
import importlib

from pyspark.sql import functions as F
from pyspark.sql import Window

# Notebook notebooks/ altında; src repo kökünde
repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
import silver_merge

# silver_merge, silver_transforms'u kullanıyor: önce o reload edilir
importlib.reload(silver_transforms)
importlib.reload(silver_merge)

print("src_path:", src_path)
print("CDC_TABLES:", silver_transforms.CDC_TABLES)

## Landing batch'ini okuma

**Amaç:** `silver_merge.read_batch` ile card batch 2'yi okumak (explicit schema, `change_ts` timestamp).

**Çıktı:** `op_type` dağılımı I 5, U 9, D 2 (toplam 16) ve `null_change_ts` 0 olmalı.

In [0]:
df_card_batch = silver_merge.read_batch(spark, "card", 2)

df_card_batch.groupBy("op_type").count().orderBy("op_type").show()

null_change_ts = df_card_batch.filter(F.col("change_ts").isNull()).count()
print("rows:", df_card_batch.count(), "| null_change_ts:", null_change_ts)

## Dedup (adım adım)

**Amaç:** Batch içinde key başına tek satır bırakmak. MERGE'te bir target satırına birden fazla source satırı eşleşirse MERGE hata verir.

| Adım | Kural |
|---|---|
| step01 | Key başına en yeni `change_ts` → `rn = 1` |
| step02 | Key'in batch'te I kaydı var mı → `has_insert` |
| step03 | Sadece `rn = 1` kalır |
| dedup | I → U ise `op_type` = I (kart target'ta yok, son haliyle insert) |

**Çıktı:** 16 satır → 15 satır / 15 key. `card_id` 1253: tek satır, `op_type` I, `type` gold.

In [0]:
from pyspark.sql import Window

#df_card_batch = silver_merge.read_batch(spark, "card", 2)
#df_card_batch.filter(F.col("card_id") == "1253").show()
# df_card_batch hücre 4'ten geliyor (batch 2)

df_card_batch_step01 = df_card_batch.withColumn("rn", 
                                                F.row_number().over(Window.partitionBy("card_id").orderBy(F.desc("change_ts")))# SQL: ROW_NUMBER() OVER (PARTITION BY card_id ORDER BY change_ts DESC)
                                               ) 
df_card_batch_step01.filter(F.col("card_id") == "1253").show()

is_insert=F.when(F.col("op_type") =="I", F.lit(1)).otherwise(F.lit(0))
has_insert=F.max(is_insert).over(Window.partitionBy("card_id")) # SQL: MAX(CASE WHEN op_type = 'I' THEN 1 ELSE 0 END) OVER (PARTITION BY card_id)
df_card_batch_step02 = df_card_batch_step01.withColumn("has_insert", has_insert)
#df_card_batch_step02.filter(F.col("card_id") == "1253").show()

df_card_batch_step02_rn=df_card_batch_step02.filter(F.col("rn") == 1)
#df_card_batch_step02_rn.filter(F.col("card_id") == "1253").show()
df_card_batch_step03 = df_card_batch_step02_rn.withColumn("op_type", F.when((F.col("has_insert")==1) & (F.col("op_type") !="D"),"I").otherwise(F.col("op_type")))
#df_card_batch_step03.filter(F.col("card_id") == "1253").show()
df_card_batch_step04=df_card_batch_step03.drop("rn", "has_insert")

df_card_dedup = df_card_batch_step04
df_card_dedup.filter(F.col("card_id") == "1253").show()
print("Distinct_keys: ",df_card_dedup.select("card_id").distinct().count())
print("Total_keys: ",df_card_dedup.count())
df_card_dedup.groupBy("op_type").count().show()


## SOURCE hazırlama ve target kontrolü

**Amaç:** Dedup sonucunu silver tiplerine çevirmek, CDC kolonlarını key ile geri join etmek (`df_card_source`) ve her source key'inin target'ta olup olmadığını göstermek (`in_target`).

**Not:** Silver şu an batch 3 sonrası durumda; bu yüzden `in_target` sayıları S2'deki ilk çalıştırmadan farklı çıkabilir. Bu bölüm sadece okur, hiçbir şey yazmaz.

In [0]:
# to_silver CDC kolonlarını atar; bu kolonlar ayrı tutulur
df_card_data = silver_transforms.to_silver(df_card_dedup, "card")
df_card_cdc = df_card_dedup.select(
    F.col("card_id").cast("int").alias("card_id"),
    "op_type",
    "change_ts",
    "batch_id",
)

# Join güvenli: dedup sonrası key başına tek satır
df_card_source = df_card_data.join(df_card_cdc, on="card_id", how="inner")

# TARGET: silver'daki mevcut key'ler
df_card_target = spark.table("workspace.silver.card")
df_target_keys = df_card_target.select("card_id").withColumn("in_target", F.lit(True))

# Left join: target'ta olmayan key'lerde in_target NULL → false
df_check = df_card_source.join(df_target_keys, on="card_id", how="left")
df_check = df_check.withColumn("in_target", F.coalesce(F.col("in_target"), F.lit(False)))

df_check.groupBy("op_type", "in_target").count().orderBy("op_type", "in_target").show()
print("target rows:", df_card_target.count())

## Card MERGE SQL'i

**Amaç:** Runner'ın kullandığı card MERGE'ünü görmek. SQL `src/silver_merge.py` içindeki `MERGE_SQL` dictionary'sinde.

| Kol | Koşul | İşlem |
|---|---|---|
| WHEN MATCHED | `op_type = 'D'` | DELETE |
| WHEN MATCHED | `op_type = 'U'` ve `s.change_ts > t._change_ts` | UPDATE |
| WHEN NOT MATCHED | `op_type = 'I'` | INSERT |

Hiçbir kola uymayan satır atlanır; target'ta olmayan key'e gelen U / D runner'da quarantine'e yazılır.

**Çıktı:** SQL metni. Bu hücre MERGE çalıştırmaz.

In [0]:
print(silver_merge.MERGE_SQL["card"])

## Batch log ve quarantine tabloları

`workspace.silver.batch_log`: her (tablo, batch) için bir satır; status STARTED / SUCCESS / FAILED. SUCCESS olan batch tekrar gelirse MERGE'e girmez.

`workspace.silver.quarantine`: target'ta olmayan key'e gelen U / D kayıtları; sayısı log'da `rows_skipped`.

**Çıktı:** `SHOW TABLES` listesinde `batch_log` ve `quarantine`. `IF NOT EXISTS` ile tekrar çalıştırılabilir.

In [0]:
%sql
-- Batch seviyesi koruma: key = table_name + batch_id
CREATE TABLE IF NOT EXISTS workspace.silver.batch_log (
  table_name    STRING,
  batch_id      INT,
  status        STRING,
  rows_read     BIGINT,
  rows_inserted BIGINT,
  rows_updated  BIGINT,
  rows_deleted  BIGINT,
  rows_skipped  BIGINT,
  started_at    TIMESTAMP,
  finished_at   TIMESTAMP
);

-- Target'ta olmayan key'e gelen U / D kayıtları
CREATE TABLE IF NOT EXISTS workspace.silver.quarantine (
  table_name STRING,
  batch_id   INT,
  key_value  INT,
  op_type    STRING,
  reason     STRING,
  change_ts  TIMESTAMP
);

SHOW TABLES IN workspace.silver;

## Runner: batch'leri log korumalı uygula

**Amaç:** `RUN_BATCH_IDS` listesindeki batch'leri sırayla (1 → 2 → 3), her batch için 4 tabloyu `silver_merge.run_batch` ile uygulamak.

Akış: log kontrolü (SUCCESS → SKIPPED) → STARTED → quarantine (`left_anti`) → MERGE → SUCCESS (hata → FAILED).

**Çıktı:** Her (tablo, batch) için SUCCESS ya da SKIPPED satırı, altında `batch_log`. Şu an 12 batch SUCCESS olduğu için 12 satırın hepsi SKIPPED çıkmalı.

In [0]:
RUN_BATCH_IDS = [1, 2, 3]

# Dış döngü batch sırası, iç döngü tablolar
for batch_id in RUN_BATCH_IDS:
    for table_name in silver_transforms.CDC_TABLES:
        silver_merge.run_batch(spark, table_name, batch_id)

display(spark.sql(f"""
    SELECT table_name, batch_id, status, rows_read, rows_updated, rows_deleted,
           rows_inserted, rows_skipped, started_at, finished_at
    FROM {silver_merge.LOG_TABLE}
    ORDER BY batch_id, table_name
"""))

## Özet

**Amaç:** Silver tablolarının son durumu. `changed_rows` = `_change_ts` > 1998-12-31 olan, yani en az bir kez MERGE ile yazılmış satırlar.

**Çıktı (batch 3 sonrası):** client 5.383 / 46, account 4.500 / 36, card 890 / 29, loan 682 / 24; `max_change_ts` Mart 1999.

In [0]:
%sql
SELECT 'client' AS table_name, COUNT(*) AS target_rows,
       COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00') AS changed_rows,
       MAX(_change_ts) AS max_change_ts
FROM workspace.silver.client
UNION ALL
SELECT 'account', COUNT(*), COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00'), MAX(_change_ts)
FROM workspace.silver.account
UNION ALL
SELECT 'card', COUNT(*), COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00'), MAX(_change_ts)
FROM workspace.silver.card
UNION ALL
SELECT 'loan', COUNT(*), COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00'), MAX(_change_ts)
FROM workspace.silver.loan;